# System 3: Damped Oscillator PINN with Model Persistence

This notebook demonstrates a Physics-Informed Neural Network for a mass-spring-damper system with the ability to save and load trained models.

**Interactive Mode**: Use the sliders below to adjust physical parameters and train new models or load existing ones.

## Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import pandas as pd
import os
from ipywidgets import interact, FloatSlider, Button, Output, VBox, HBox, Checkbox
from IPython.display import display, clear_output

## 1. Interactive Physical Parameters

Adjust the sliders below to modify the physical parameters of the system.

In [ ]:
# Define sliders for physical parameters
m_slider = FloatSlider(value=10.0, min=1.0, max=50.0, step=0.5, description='Mass [kg]:', style={'description_width': 'initial'})
k_slider = FloatSlider(value=200.0, min=50.0, max=1000.0, step=10.0, description='Stiffness [N/m]:', style={'description_width': 'initial'})
c_slider = FloatSlider(value=15.0, min=1.0, max=100.0, step=1.0, description='Damping [Ns/m]:', style={'description_width': 'initial'})
g_slider = FloatSlider(value=9.81, min=0.0, max=20.0, step=0.1, description='Gravity [m/s²]:', style={'description_width': 'initial'})
x0_slider = FloatSlider(value=1.2, min=-5.0, max=5.0, step=0.1, description='Initial Position [m]:', style={'description_width': 'initial'})
v0_slider = FloatSlider(value=3.0, min=-10.0, max=10.0, step=0.1, description='Initial Velocity [m/s]:', style={'description_width': 'initial'})
t_max_slider = FloatSlider(value=5.0, min=1.0, max=20.0, step=0.5, description='Simulation Time [s]:', style={'description_width': 'initial'})

# Checkbox for forcing retraining
force_retrain = Checkbox(value=False, description='Force Retrain (ignore saved model)', style={'description_width': 'initial'})

# Display sliders
print("="*60)
print("PHYSICAL PARAMETERS - Adjust sliders below:")
print("="*60)
display(m_slider)
display(k_slider)
display(c_slider)
display(g_slider)
display(x0_slider)
display(v0_slider)
display(t_max_slider)
display(force_retrain)

## 2. PINN Architecture

In [ ]:
class VibrationPINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 40), nn.Tanh(),
            nn.Linear(40, 40), nn.Tanh(),
            nn.Linear(40, 40), nn.Tanh(),
            nn.Linear(40, 3)  # [x, y, lambda]
        )

    def forward(self, t):
        return self.net(t)

## 3. Training and Model Management

In [ ]:
# Output widget for displaying results
results_output = Output()

def get_physics_loss(model, t_collocation, m, k, c, g):
    """Physics loss function with parameterized physical constants."""
    t_collocation.requires_grad = True
    pred = model(t_collocation)
    x, y, lam = pred[:, 0:1], pred[:, 1:2], pred[:, 2:3]
    dx_dt = torch.autograd.grad(x, t_collocation, torch.ones_like(x), create_graph=True)[0]
    d2x_dt2 = torch.autograd.grad(dx_dt, t_collocation, torch.ones_like(dx_dt), create_graph=True)[0]
    dy_dt = torch.autograd.grad(y, t_collocation, torch.ones_like(y), create_graph=True)[0]
    d2y_dt2 = torch.autograd.grad(dy_dt, t_collocation, torch.ones_like(dy_dt), create_graph=True)[0]

    res_x = m * d2x_dt2 + c * dx_dt + k * x
    res_y = m * d2y_dt2 + lam + m * g
    res_phi = y
    return torch.mean(res_x**2) + torch.mean(res_y**2) + 10 * torch.mean(res_phi**2)

def get_model_filename(m, k, c, g, x0, v0, t_max):
    """Generate a unique filename based on parameters."""
    return f"pinn_m{m}_k{k}_c{c}_g{g}_x0{x0}_v0{v0}_t{t_max}.pth"

def train_and_save_model(m, k, c, g, x0, v0, t_max, force_retrain=False):
    """
    Train a new PINN model or load existing one.
    Returns the trained model and training info.
    """
    MODEL_PATH = get_model_filename(m, k, c, g, x0, v0, t_max)
    pinn = VibrationPINN()
    trained_new = False
    
    if os.path.exists(MODEL_PATH) and not force_retrain:
        print(f"\n--- Found saved model: {MODEL_PATH} ---")
        print("Loading weights... Skipping training.")
        pinn.load_state_dict(torch.load(MODEL_PATH))
        pinn.eval()
    else:
        if force_retrain:
            print(f"\n--- Force retraining enabled ---")
        print(f"\n--- No saved model found. Commencing Training... ---")
        print(f"Model will be saved as: {MODEL_PATH}")
        
        optimizer = torch.optim.Adam(pinn.parameters(), lr=1e-3)
        t_train = torch.linspace(0, t_max, 600).view(-1, 1)

        for epoch in range(12001):
            optimizer.zero_grad()
            loss_p = get_physics_loss(pinn, t_train, m, k, c, g)

            t0 = torch.tensor([[0.0]], requires_grad=True)
            p0 = pinn(t0)
            x0_p = p0[:, 0:1]
            v0_p = torch.autograd.grad(x0_p, t0, torch.ones_like(x0_p), create_graph=True)[0]
            loss_ic = (x0_p - x0)**2 + (v0_p - v0)**2 + (p0[:, 1:2])**2

            total_loss = loss_p + 50 * loss_ic
            total_loss.backward()
            optimizer.step()

            if epoch % 3000 == 0:
                print(f"Epoch {epoch:5d} | Loss: {total_loss.item():.8f}")

        # SAVE THE MODEL
        torch.save(pinn.state_dict(), MODEL_PATH)
        print(f"Training complete. Model saved to {MODEL_PATH}")
        trained_new = True
    
    return pinn, MODEL_PATH, trained_new

def run_with_parameters(btn=None):
    """
    Run the complete pipeline with current slider values.
    """
    with results_output:
        clear_output(wait=True)
        
        # Get current slider values
        m = m_slider.value
        k = k_slider.value
        c = c_slider.value
        g = g_slider.value
        x0 = x0_slider.value
        v0 = v0_slider.value
        t_max = t_max_slider.value
        retrain = force_retrain.value
        
        # Calculate system properties
        wn = np.sqrt(k/m)
        zeta = c / (2 * np.sqrt(m*k))
        if zeta < 1:
            damping_type = "Underdamped"
        elif zeta == 1:
            damping_type = "Critically Damped"
        else:
            damping_type = "Overdamped"
        
        print(f"\n--- System Parameters ---")
        print(f"Mass: {m} kg | Stiffness: {k} N/m | Damping: {c} Ns/m")
        print(f"Natural Freq: {wn:.2f} rad/s | Damping Ratio: {zeta:.3f} ({damping_type})")
        
        # Train or load model
        pinn, model_path, trained_new = train_and_save_model(m, k, c, g, x0, v0, t_max, retrain)
        
        # Numerical benchmark
        def mbd_ground_truth(t, state):
            x, v = state
            return [v, (-k*x - c*v) / m]
        
        t_eval = np.linspace(0, t_max, 200)
        sol = solve_ivp(mbd_ground_truth, [0, t_max], [x0, v0], t_eval=t_eval)
        
        # PINN predictions
        t_test = torch.tensor(t_eval, dtype=torch.float32).view(-1, 1)
        with torch.no_grad():
            results = pinn(t_test)
            x_pinn = results[:, 0].numpy()
        
        # Comparison table
        df_comp = pd.DataFrame({
            "Time (s)": t_eval[::20],
            "RK45 x(t)": sol.y[0][::20],
            "PINN x(t)": x_pinn[::20],
            "Error": np.abs(sol.y[0][::20] - x_pinn[::20])
        })
        print("\n--- PERFORMANCE VALIDATION ---")
        print(df_comp.to_string(index=False))
        
        # Plotting
        fig, axs = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle(f"System 3: Model Validation (m={m}kg, k={k}N/m, c={c}Ns/m)\nζ={zeta:.3f} ({damping_type})", fontsize=12)
        
        axs[0].plot(t_eval, sol.y[0], 'k--', label='RK45 (Numerical)', alpha=0.5)
        axs[0].plot(t_eval, x_pinn, 'b-', label='PINN (Stored Model)')
        axs[0].set_title("Displacement Comparison")
        axs[0].set_xlabel("Time [s]")
        axs[0].set_ylabel("Position [m]")
        axs[0].legend()
        axs[0].grid(True)
        
        # Phase portrait
        t_grad = t_test.clone().requires_grad_(True)
        x_out = pinn(t_grad)[:, 0:1]
        v_pinn = torch.autograd.grad(x_out, t_grad, torch.ones_like(x_out))[0].detach().numpy()
        
        axs[1].plot(sol.y[0], sol.y[1], 'k--', label='RK45', alpha=0.5)
        axs[1].plot(x_pinn, v_pinn, 'r-', label='PINN')
        axs[1].set_title("Phase Portrait")
        axs[1].set_xlabel("Position [m]")
        axs[1].set_ylabel("Velocity [m/s]")
        axs[1].legend()
        axs[1].grid(True)
        
        plt.tight_layout()
        plt.show()
        
        print(f"\nModel file: {model_path}")

# Create run button
run_button = Button(description="🚀 Train/Load PINN Model", button_style='success', layout={'width': '250px'})
run_button.on_click(run_with_parameters)

print("\n" + "="*60)
print("Click the button below to train or load a model:")
print("="*60)
display(run_button)
display(results_output)

## 4. Quick Exploration (Numerical Only)

In [ ]:
def quick_explore_persistent(m=10.0, k=200.0, c=15.0, g=9.81, x0=1.2, v0=3.0, t_max=5.0):
    """
    Quick exploration using numerical solution only.
    Use this to preview system behavior before training PINN.
    """
    # Vibration constants
    wn = np.sqrt(k/m)
    zeta = c / (2 * np.sqrt(m*k))
    
    if zeta < 1:
        damping_type = "Underdamped"
        wd = wn * np.sqrt(1 - zeta**2)
    elif zeta == 1:
        damping_type = "Critically Damped"
        wd = 0
    else:
        damping_type = "Overdamped"
        wd = 0
    
    # Numerical solution
    def mbd_ground_truth(t, state):
        x, v = state
        return [v, (-k*x - c*v) / m]
    
    t_eval = np.linspace(0, t_max, 500)
    sol = solve_ivp(mbd_ground_truth, [0, t_max], [x0, v0], t_eval=t_eval, method='RK45')
    
    # Plot
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Quick Preview: m={m}kg, k={k}N/m, c={c}Ns/m | ζ={zeta:.3f} ({damping_type})", fontsize=12)
    
    axs[0].plot(t_eval, sol.y[0], 'b-', linewidth=2)
    axs[0].set_title("Displacement Response")
    axs[0].set_xlabel("Time [s]")
    axs[0].set_ylabel("Position x(t) [m]")
    axs[0].grid(True)
    axs[0].axhline(y=0, color='k', linestyle='--', alpha=0.3)
    
    axs[1].plot(sol.y[0], sol.y[1], 'r-', linewidth=2)
    axs[1].set_title("Phase Portrait")
    axs[1].set_xlabel("Displacement [m]")
    axs[1].set_ylabel("Velocity [m/s]")
    axs[1].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nSystem Properties:")
    print(f"  Natural Frequency (ωn): {wn:.2f} rad/s ({wn/(2*np.pi):.2f} Hz)")
    print(f"  Damping Ratio (ζ): {zeta:.3f} ({damping_type})")
    if zeta < 1:
        print(f"  Damped Frequency (ωd): {wd:.2f} rad/s ({wd/(2*np.pi):.2f} Hz)")
        print(f"  Period: {2*np.pi/wd:.3f} s")
    
    # Show what filename would be
    model_name = get_model_filename(m, k, c, g, x0, v0, t_max)
    exists = os.path.exists(model_name)
    print(f"\nModel file: {model_name}")
    print(f"File exists: {'Yes ✓' if exists else 'No (will need to train)'}")

# Create interactive widget
print("\n" + "="*60)
print("QUICK EXPLORATION (Numerical Solution Only)")
print("="*60)
interact(quick_explore_persistent,
         m=FloatSlider(value=10.0, min=1.0, max=50.0, step=0.5, description='Mass [kg]:'),
         k=FloatSlider(value=200.0, min=50.0, max=1000.0, step=10.0, description='Stiffness [N/m]:'),
         c=FloatSlider(value=15.0, min=1.0, max=100.0, step=1.0, description='Damping [Ns/m]:'),
         g=FloatSlider(value=9.81, min=0.0, max=20.0, step=0.1, description='Gravity [m/s²]:'),
         x0=FloatSlider(value=1.2, min=-5.0, max=5.0, step=0.1, description='Init Pos [m]:'),
         v0=FloatSlider(value=3.0, min=-10.0, max=10.0, step=0.1, description='Init Vel [m/s]:'),
         t_max=FloatSlider(value=5.0, min=1.0, max=20.0, step=0.5, description='Time [s]:'));

## 5. List Saved Models

In [ ]:
def list_saved_models():
    """List all saved PINN models in the current directory."""
    print("\n--- Saved PINN Models ---")
    pth_files = [f for f in os.listdir('.') if f.endswith('.pth')]
    if pth_files:
        for f in sorted(pth_files):
            size_kb = os.path.getsize(f) / 1024
            print(f"  {f} ({size_kb:.1f} KB)")
    else:
        print("  No saved models found.")
    return pth_files

list_saved_models()